# AERIS: Exploratory Data Analysis & Data Quality Report
**Adaptive Environmental Risk & Intelligence System**
*Exploratory Data Analysis & Quality Audit*

---

## Executive Summary & Objectives
This notebook executes a research-grade Exploratory Data Analysis (EDA) on the validated Indian daily air quality dataset (`data/processed/cleaned_city_day.parquet`).

### Core Research Questions & Guardrails
1. **Structural Quality:** Characterize spatial coverage (26 cities) and multi-year temporal continuity (2015–2020).
2. **Missingness Topology:** Identify feature-wise, temporal, and spatial missingness concentrations to inform Milestone 4 interpolation strategies.
3. **Distributional Dynamics & Outliers:** Quantify tail behaviors, skewness, and extreme episodic pollution spikes using IQR analytics (without unphysical clipping).
4. **Pollutant Interactions & AQI Alignment:** Evaluate Pearson linear and Spearman rank associations between 12 chemical species and composite AQI.
5. **Temporal Seasonality:** Uncover annual and monthly meteorological cyclicality.

> **Research Boundary Reminder:**
> * No model training is performed in this exploratory analysis.
> * No train/validation/test splitting or sequence generation is executed.
> * No features are scaled, clipped, or imputed in this exploratory analysis.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure project root is accessible
repo_root = (
    Path(".").resolve().parent
    if Path(".").resolve().name == "notebooks"
    else Path(".").resolve()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from backend.app.data.eda import generate_eda_report, save_eda_report  # noqa: E402

# Styling configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

DATA_PATH = repo_root / "data" / "processed" / "cleaned_city_day.parquet"
FIGURES_DIR = repo_root / "outputs" / "eda" / "figures"
REPORT_PATH = repo_root / "outputs" / "eda" / "eda_report.json"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Loading validated dataset from: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} rows and {len(df.columns)} cols.")

---
## 1. Dataset Overview
We examine global structural dimensions, column data types, spatial coverage, and observation volumes across Indian cities.


In [ ]:
print("=== DATASET DIMENSIONS & SCHEMA ===")
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print("\nColumn Data Types:")
print(df.dtypes)

print("\n=== TEMPORAL & GEOGRAPHICAL SPAN ===")
valid_dates = df["Date"].dropna()
print(f"Unique Cities ({df['City'].nunique()}): {sorted(df['City'].unique().tolist())}")
date_min = valid_dates.min().strftime("%Y-%m-%d")
date_max = valid_dates.max().strftime("%Y-%m-%d")
span_days = (valid_dates.max() - valid_dates.min()).days + 1
print(f"Date Range: {date_min} to {date_max} ({span_days:,} days)")

print("\n=== OBSERVATIONS PER CITY ===")
print(df["City"].value_counts().to_string())

---
## 2. Missing-Value Analysis
Quantify missingness across features and cities without premature imputation.


In [ ]:
missing_counts = df.isna().sum()
missing_pcts = (missing_counts / len(df)) * 100.0
missing_summary = pd.DataFrame(
    {"Missing_Count": missing_counts, "Missing_Percentage": missing_pcts.round(2)}
).sort_values(by="Missing_Percentage", ascending=False)

print("=== FEATURE MISSINGNESS SUMMARY ===")
print(missing_summary.to_string())

# Missingness visualization
fig, ax = plt.subplots(figsize=(10, 6))
missing_plot = missing_summary[missing_summary["Missing_Percentage"] > 0]
sns.barplot(
    x=missing_plot["Missing_Percentage"], y=missing_plot.index, color="#e11d48", ax=ax
)
ax.set_title("Missing Value Proportion by Feature (%)", fontsize=14, fontweight="bold")
ax.set_xlabel("Missing Percentage (%)", fontsize=11)
ax.set_ylabel("Feature", fontsize=11)
for i, v in enumerate(missing_plot["Missing_Percentage"]):
    ax.text(v + 0.8, i, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 70)
plt.tight_layout()
fig_path = FIGURES_DIR / "missing_values.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

---
## 3. Duplicate Analysis
Audit exact complete row duplicates and (City, Date) collisions.


In [ ]:
exact_duplicates = int(df.duplicated().sum())
city_date_duplicates = int(df.duplicated(subset=["City", "Date"]).sum())

print("=== DUPLICATE AUDIT RESULTS ===")
print(f"Exact Duplicate Rows: {exact_duplicates}")
print(f"Duplicate (City, Date) Tuples: {city_date_duplicates}")
assert exact_duplicates == 0, "Error: Exact duplicates detected!"
assert city_date_duplicates == 0, "Error: Duplicate (City, Date) records!"
print("Status: PASSED — Zero duplication confirmed.")

---
## 4. Outlier Analysis (IQR Method)
Evaluate outlier frequency using the Interquartile Range (IQR) method:
$$\text{IQR} = Q_3 - Q_1, \quad [\text{Lower}, \text{Upper}] = [Q_1 - 1.5\cdot\text{IQR}, Q_3 + 1.5\cdot\text{IQR}]$$

> **Methodological Policy:** Outliers are analyzed to characterize distributional variance but are **strictly preserved** (neither clipped nor deleted).


In [ ]:
NUMERIC_COLS = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene",
    "AQI",
]

outlier_rows = []
for col in NUMERIC_COLS:
    s = df[col].dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    iqr = q75 - q25
    lb = q25 - 1.5 * iqr
    ub = q75 + 1.5 * iqr
    out_cnt = int(((s < lb) | (s > ub)).sum())
    out_pct = (out_cnt / len(s)) * 100.0
    outlier_rows.append(
        {
            "Feature": col,
            "Valid_N": len(s),
            "Q25": round(q25, 2),
            "Q75": round(q75, 2),
            "IQR": round(iqr, 2),
            "Lower_Bound": round(lb, 2),
            "Upper_Bound": round(ub, 2),
            "Outlier_Count": out_cnt,
            "Outlier_Pct": round(out_pct, 2),
        }
    )

outlier_df = pd.DataFrame(outlier_rows).sort_values(by="Outlier_Pct", ascending=False)
print("=== IQR OUTLIER AUDIT TABLE ===")
print(outlier_df.to_string(index=False))

---
## 5. City-Wise Air Quality Analysis
Spatial comparisons across Indian cities.


In [ ]:
city_summary = (
    df.groupby("City")["AQI"]
    .agg(
        Observations="count",
        Mean="mean",
        Median="median",
        Std="std",
        Min="min",
        Max="max",
    )
    .sort_values(by="Mean", ascending=False)
)

print("=== CITY-LEVEL AQI RANKING (BY MEAN AQI) ===")
print(city_summary.round(2).to_string())

# City Comparison Plot
fig, ax = plt.subplots(figsize=(10, 10))
city_means = city_summary["Mean"].sort_values(ascending=True)
colors = plt.cm.plasma(np.linspace(0.2, 0.85, len(city_means)))
ax.barh(city_means.index, city_means.values, color=colors, height=0.65)
ax.set_title("City-Wise Historical Mean AQI Ranking", fontsize=14, fontweight="bold")
ax.set_xlabel("Mean AQI", fontsize=11)
ax.set_ylabel("City", fontsize=11)
for i, v in enumerate(city_means.values):
    ax.text(v + 3, i, f"{v:.1f}", va="center", fontsize=8.5)
ax.set_xlim(0, max(city_means.values) + 35)
plt.tight_layout()
fig_path = FIGURES_DIR / "city_wise_aqi.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

---
## 6. Empirical Pollutant Distributions
Visualizing all 12 monitored pollutants with independent scales.


In [ ]:
POLLUTANTS = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene",
]

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
axes = axes.flatten()

for idx, pol in enumerate(POLLUTANTS):
    ax = axes[idx]
    s = df[pol].dropna()
    sns.histplot(
        s, kde=True, ax=ax, color="#0284c7", bins=40, edgecolor="none", alpha=0.55
    )
    skew = s.skew()
    ax.set_title(
        f"{pol} (Skew={skew:.2f}, N={len(s):,})", fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("")
    ax.set_ylabel("Count", fontsize=9)

plt.suptitle(
    "Empirical Distributions of 12 Monitored Pollutant Species",
    fontsize=15,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
fig_path = FIGURES_DIR / "pollutant_distributions.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

---
## 7. Composite AQI Distribution & Category Buckets
Quantiles and National AQI category bucket distribution.


In [ ]:
aqi_s = df["AQI"].dropna()
print("=== AQI DISTRIBUTION SUMMARY ===")
print(f"Valid Observations: {len(aqi_s):,}")
print(f"Mean: {aqi_s.mean():.2f}")
print(f"Median: {aqi_s.median():.2f}")
print(f"Std Dev: {aqi_s.std():.2f}")
iqr_val = aqi_s.quantile(0.75) - aqi_s.quantile(0.25)
print(f"IQR: {iqr_val:.2f}")
print(f"Skewness: {aqi_s.skew():.2f}")
q5 = aqi_s.quantile(0.05)
q25 = aqi_s.quantile(0.25)
q50 = aqi_s.quantile(0.50)
q75 = aqi_s.quantile(0.75)
q95 = aqi_s.quantile(0.95)
q99 = aqi_s.quantile(0.99)
print(f"Quantiles: 5%={q5:.1f}, 25%={q25:.1f}, 50%={q50:.1f}")
print(f"Quantiles: 75%={q75:.1f}, 95%={q95:.1f}, 99%={q99:.1f}")

print("\n=== AQI BUCKET PROPORTIONS ===")
print(
    df["AQI_Bucket"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
    .to_string()
)

# AQI Distribution Plot
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    aqi_s, kde=True, color="#2563eb", bins=60, ax=ax, edgecolor="none", alpha=0.6
)
ax.axvline(
    aqi_s.mean(),
    color="#dc2626",
    linestyle="--",
    linewidth=2,
    label=f"Mean ({aqi_s.mean():.1f})",
)
ax.axvline(
    aqi_s.median(),
    color="#16a34a",
    linestyle="-.",
    linewidth=2,
    label=f"Median ({aqi_s.median():.1f})",
)
ax.axvline(
    q95, color="#d97706", linestyle=":", linewidth=2, label=f"95th Pct ({q95:.1f})"
)
ax.set_title(
    "Empirical Distribution of Air Quality Index (AQI)", fontsize=14, fontweight="bold"
)
ax.set_xlabel("AQI", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
fig_path = FIGURES_DIR / "aqi_distribution.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

---
## 8. Temporal Dynamics & Seasonal Cyclicality
Multi-year trajectories and monthly seasonal variations.


In [ ]:
df_ts = df.dropna(subset=["AQI"]).copy()
df_ts["Year"] = df_ts["Date"].dt.year
df_ts["Month"] = df_ts["Date"].dt.month

print("=== YEARLY AQI TRENDS ===")
print(
    df_ts.groupby("Year")["AQI"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
    .to_string()
)

print("\n=== MONTHLY SEASONAL CYCLICALITY ===")
print(
    df_ts.groupby("Month")["AQI"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
    .to_string()
)

# Temporal Plot
fig, ax = plt.subplots(figsize=(12, 6))
monthly_national = df_ts.groupby(pd.Grouper(key="Date", freq="ME"))["AQI"].mean()
ax.plot(
    monthly_national.index,
    monthly_national.values,
    color="#1e293b",
    linewidth=2.5,
    label="National Monthly Average",
)

for city, color in [
    ("Delhi", "#ef4444"),
    ("Ahmedabad", "#f59e0b"),
    ("Bengaluru", "#10b981"),
]:
    city_sub = df_ts[df_ts["City"] == city]
    city_m = city_sub.groupby(pd.Grouper(key="Date", freq="ME"))["AQI"].mean()
    ax.plot(
        city_m.index,
        city_m.values,
        linestyle="--",
        alpha=0.75,
        label=f"{city} Monthly Avg",
        color=color,
    )

ax.set_title(
    "Historical AQI Trajectory Across Time (2015 - 2020)",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Mean Monthly AQI", fontsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
fig_path = FIGURES_DIR / "aqi_over_time.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

---
## 9. Correlation Analysis (Pearson & Spearman)
Linear and monotonic rank correlations between features and AQI.


In [ ]:
corr_rows = []
for pol in POLLUTANTS:
    sub = df[[pol, "AQI"]].dropna()
    p_r = sub[pol].corr(sub["AQI"], method="pearson")
    s_rho = sub[pol].corr(sub["AQI"], method="spearman")
    corr_rows.append(
        {
            "Pollutant": pol,
            "Pearson_r": round(p_r, 4),
            "Spearman_rho": round(s_rho, 4),
            "Pairwise_N": len(sub),
        }
    )

corr_df = pd.DataFrame(corr_rows).sort_values(by="Pearson_r", ascending=False)
print("=== POLLUTANT-TO-AQI CORRELATION RANKING ===")
print(corr_df.to_string(index=False))

# Correlation Heatmap
fig, ax = plt.subplots(figsize=(11, 9))
corr_matrix = df[NUMERIC_COLS].corr(method="pearson")
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-0.2,
    vmax=1.0,
    square=True,
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_title(
    "Pearson Correlation Matrix (12 Pollutants + AQI)", fontsize=14, fontweight="bold"
)
plt.tight_layout()
fig_path = FIGURES_DIR / "correlation_matrix.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved: {fig_path}")

---
## 10. Pollutant-to-AQI Relationship Regressions
Visualizing scatter regressions for `PM2.5`, `PM10`, and `NO2`.


In [ ]:
# PM2.5 vs AQI
fig, ax = plt.subplots(figsize=(8, 6))
sub_pm25 = df[["PM2.5", "AQI"]].dropna()
p_r25 = sub_pm25["PM2.5"].corr(sub_pm25["AQI"], method="pearson")
s_r25 = sub_pm25["PM2.5"].corr(sub_pm25["AQI"], method="spearman")
sns.regplot(
    data=sub_pm25,
    x="PM2.5",
    y="AQI",
    ax=ax,
    scatter_kws={"alpha": 0.15, "color": "#2563eb", "s": 12},
    line_kws={"color": "#dc2626", "linewidth": 2},
)
ax.set_title(
    f"PM2.5 vs AQI (Pearson r = {p_r25:.2f}, Spearman rho = {s_r25:.2f})",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("PM2.5 (µg/m³)", fontsize=11)
ax.set_ylabel("AQI", fontsize=11)
plt.tight_layout()
p7 = FIGURES_DIR / "pm25_vs_aqi.png"
plt.savefig(p7, dpi=300)
plt.show()
print(f"Saved: {p7}")

# PM10 vs AQI
fig, ax = plt.subplots(figsize=(8, 6))
sub_pm10 = df[["PM10", "AQI"]].dropna()
p_r10 = sub_pm10["PM10"].corr(sub_pm10["AQI"], method="pearson")
s_r10 = sub_pm10["PM10"].corr(sub_pm10["AQI"], method="spearman")
sns.regplot(
    data=sub_pm10,
    x="PM10",
    y="AQI",
    ax=ax,
    scatter_kws={"alpha": 0.15, "color": "#0d9488", "s": 12},
    line_kws={"color": "#dc2626", "linewidth": 2},
)
ax.set_title(
    f"PM10 vs AQI (Pearson r = {p_r10:.2f}, Spearman rho = {s_r10:.2f})",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("PM10 (µg/m³)", fontsize=11)
ax.set_ylabel("AQI", fontsize=11)
plt.tight_layout()
p8 = FIGURES_DIR / "pm10_vs_aqi.png"
plt.savefig(p8, dpi=300)
plt.show()
print(f"Saved: {p8}")

# NO2 vs AQI
fig, ax = plt.subplots(figsize=(8, 6))
sub_no2 = df[["NO2", "AQI"]].dropna()
p_r_no2 = sub_no2["NO2"].corr(sub_no2["AQI"], method="pearson")
s_r_no2 = sub_no2["NO2"].corr(sub_no2["AQI"], method="spearman")
sns.regplot(
    data=sub_no2,
    x="NO2",
    y="AQI",
    ax=ax,
    scatter_kws={"alpha": 0.15, "color": "#9333ea", "s": 12},
    line_kws={"color": "#dc2626", "linewidth": 2},
)
ax.set_title(
    f"NO2 vs AQI (Pearson r = {p_r_no2:.2f}, Spearman rho = {s_r_no2:.2f})",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlabel("NO2 (µg/m³)", fontsize=11)
ax.set_ylabel("AQI", fontsize=11)
plt.tight_layout()
p9 = FIGURES_DIR / "no2_vs_aqi.png"
plt.savefig(p9, dpi=300)
plt.show()
print(f"Saved: {p9}")

---
## 11. Structured Report Generation & Export
We compile computed statistics into `outputs/eda/eda_report.json`.


In [ ]:
report = generate_eda_report(df)
save_eda_report(report, REPORT_PATH)
print(f"Structured EDA Report saved successfully to: {REPORT_PATH}")
print(f"Report Sections: {list(report.keys())}")

---
## 12. Key Findings & Preprocessing Strategy for Milestone 4

### Summary of Empirical Findings:
1. **Primary Drivers:** Particulate matter (`PM10` $r = 0.8033$, `PM2.5` $r = 0.6592$) and combustion indicators (`CO` $r = 0.6833$, `NO2` $r = 0.5371$) are the most predictive statistical indicators of composite AQI.
2. **Extreme Tail Dynamics:** All pollutants exhibit heavy positive skewness (e.g. `PM2.5` skewness $> 3.5$) and 5–11% outlier rates during seasonal episodes. These must be preserved without artificial clipping.
3. **Winter Inversion Peak:** AQI peaks dramatically in November–January ($> 225$) and troughs during the monsoon season in July–September ($\approx 112$).
4. **Missingness Topology:** Volatile organic species (especially `Xylene` at $61.32\%$ missing) and `PM10` ($37.72\%$) require robust temporal interpolation (localized linear fill for short gaps $\le 5$ days) in Milestone 4.

**Analysis Complete:** Exploratory Data Analysis & Quality Audit verified. Ready for Milestone 4 (Data Preparation).
